解析pdf文件，获得脏数据，使用Mineru镜像

In [ ]:
import os 
from pathlib import Path 
import logging
from datetime import datetime
import subprocess
import shutil
import time

log_dir = Path( Path.cwd().parents[0] , "logging")
if not log_dir.exists():
    log_dir.mkdir(parents=True, exist_ok=True)
log_dir = Path( Path.cwd().parents[0] , "logging")
if not log_dir.exists():
    log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / f"pdf_parser_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
print(f"📄 log保存位置: {log_file}")
print(f"📁 log目录: {log_dir}")
p = Path( Path.cwd().parents[0], 'data' , 'raw' , '示例数据','附件2：财务报告' )
dirs_names  = [Path(p, name) for name in os.listdir(p)]
all_files = []
for dir in dirs_names:
    files = [Path(dir, name) for name in os.listdir(dir)]
    all_files.extend(files)
all_files

In [ ]:

# ===== 配置 =====
PDF_ROOT = Path("/root/TAIDIBEI_B/data/raw/示例数据/附件2：财务报告")
OUTPUT_ROOT = Path("/root/TAIDIBEI_B/parsed_results")
LOG_DIR = Path("/root/TAIDIBEI_B/logs")
# ================

# 创建目录
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 配置日志
log_file = LOG_DIR / f"processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

def process_pdf(pdf_path):
    """处理单个PDF并自动移动文件"""
    
    # 提取信息
    company = pdf_path.parent.name  # reports-上交所 或 reports-深交所
    pdf_name = pdf_path.stem  # 不带后缀的文件名
    
    # 创建目标目录
    target_dir = OUTPUT_ROOT / company
    images_dir = target_dir / "images"


    target_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)
    
    logging.info(f"处理: {pdf_path.name}")
    logging.info(f"目标: {target_dir}")
    
    # 运行magic-pdf命令
    cmd = ["magic-pdf", "pdf-command", "--pdf", str(pdf_path), "--method", "auto"]
    
    try:
        # 执行命令
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        
        if result.returncode == 0:
            # 查找临时目录中的结果
            tmp_dir = Path(f"/tmp/magic-pdf/{pdf_name}/auto")
            
            if tmp_dir.exists():
                # 移动markdown文件
                md_files = list(tmp_dir.glob("*.md"))
                for md in md_files:
                    shutil.move(str(md), str(target_dir / md.name))
                    logging.info(f"  📄 移动: {md.name}")
                
                # 移动json文件
                json_files = list(tmp_dir.glob("*.json"))
                for json in json_files:
                    shutil.move(str(json), str(target_dir / json.name))
                    logging.info(f"  📊 移动: {json.name}")
                
                # 移动图片
                img_dir = tmp_dir / "images"
                if img_dir.exists():
                    jpg_count = 0 
                    for img in img_dir.glob("*.jpg"):
                        shutil.move(str(img), str(images_dir / img.name))
                        jpg_count += 1
                    logging.info(f"  🖼️  移动图片: {jpg_count}张")
                    img_dir.rmdir()  # 删除空目录
                
                if tmp_dir.parent.exists():
                    shutil.rmtree(tmp_dir.parent) 
                    logging.info(f"  🗑️ 清理临时目录: {tmp_dir.parent}")
                
                logging.info(f"  ✅ 完成: {pdf_name}")
                return True
            else:
                logging.warning(f"  ⚠️ 未找到临时目录: {tmp_dir}")
                return False
        else:
            logging.error(f"  ❌ 处理失败: {pdf_path.name}")
            if result.stderr:
                logging.error(f"     错误: {result.stderr[:200]}")
            return False
            
    except subprocess.TimeoutExpired:
        logging.error(f"  ⏰ 超时: {pdf_path.name}")
        return False
    except Exception as e:
        logging.error(f"  💥 异常: {pdf_path.name}, {str(e)}")
        return False

def main():
    # 查找所有PDF
    all_pdfs = list(PDF_ROOT.rglob("*.pdf"))
    logging.info(f"="*60)
    logging.info(f"🚀 开始处理，共 {len(all_pdfs)} 个PDF文件")
    logging.info(f"📁 输出目录: {OUTPUT_ROOT}")
    logging.info(f"📄 日志文件: {log_file}")
    logging.info(f"="*60)
    
    success = 0
    failed = 0
    start_time = time.time()
    
    for i, pdf in enumerate(all_pdfs, 1):
        logging.info(f"\n[{i}/{len(all_pdfs)}] 进度: {i/len(all_pdfs)*100:.1f}%")
        
        if process_pdf(pdf):
            success += 1
        else:
            failed += 1
        
        # 每5个文件休息一下
        if i % 5 == 0 and i < len(all_pdfs):
            logging.info(f"⏸️  已处理 {i} 个，休息3秒...")
            time.sleep(3)
    
    # 统计
    elapsed = time.time() - start_time
    hours = int(elapsed // 3600)
    minutes = int((elapsed % 3600) // 60)
    seconds = int(elapsed % 60)
    
    logging.info("\n" + "="*60)
    logging.info(f"✅ 全部处理完成！")
    logging.info(f"📊 统计:")
    logging.info(f"   总文件: {len(all_pdfs)}")
    logging.info(f"   成功: {success}")
    logging.info(f"   失败: {failed}")
    logging.info(f"   用时: {hours}小时{minutes}分钟{seconds}秒")
    logging.info(f"📁 结果目录: {OUTPUT_ROOT}")
    logging.info(f"📄 日志文件: {log_file}")
    logging.info("="*60)

if __name__ == "__main__":
    main()

解析markdown数据，使用大模型图生文大模型进行图片解析，调用qwen3-vl-flash模型。
一个新的虚拟环境
conda create -n py10 python=3.10
conda activate py10
pip install numpy pandas matplotlib scikit-learn
pip install ipykernel jupyter

python -m ipykernel install --user --name=py10 --display-name="py10"
Installed kernelspec py10 in C:\Users\86193\AppData\Roaming\jupyter\kernels\py10

pip install python-dotenv #配置全局环境
pip install openai
pip install openpyxl
pip install tqdm

pip install zai-sdk

In [2]:
import base64
import os 
from openai import OpenAI
from pathlib import Path
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
def img_to_markdown(image_path):
    base64_image = encode_image(image_path)
    client = OpenAI(
        api_key=os.getenv("ALIYUN_API_KEY"),  # 你的 API Key
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )   
    completion = client.chat.completions.create(
        model="qwen3-vl-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    },
                    {"type": "text", "text": "请识别图片中的表格内容，并以 Markdown 格式返回。"}
                ]
            }
        ]
    )
    
    return completion.choices[0].message.content

def clear(file_dir,target_file):
    target_file.parent.mkdir(parents=True, exist_ok=True)
    if not target_file.exists():
        target_file.touch()  # 创建空文件
        print(f"文件已创建: {target_file}")
    else:
        print(f"文件已存在: {target_file}")
    with open(file_dir, "r", encoding="utf-8") as f:
        content = f.read()

    with tqdm(total=content.count('![]'), desc=f"处理中{file_dir.name}") as pbar:
        for _ in range(content.count('![]')):
            start = content.find('![]')
            end = content.find(')', start )
            replaced = content[start:end+1]
            images_dir = file_dir.parent / replaced[4:-1] # 提取图片路径
            translation = img_to_markdown(str(images_dir))
            translation = translation.replace("markdown", "")
            content = content.replace(replaced, translation)
            pbar.update(1)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    


文件已存在: e:\github\TAIDIBEI_B\processed\reports-上交所\600080_20250822_ODWZ.md


处理中600080_20250822_ODWZ.md: 100%|██████████| 220/220 [17:44<00:00,  4.84s/it]


文件已存在: e:\github\TAIDIBEI_B\processed\reports-上交所\600080_20251030_IVCB.md


处理中600080_20251030_IVCB.md: 100%|██████████| 16/16 [02:15<00:00,  8.45s/it]


文件已存在: e:\github\TAIDIBEI_B\processed\reports-深交所\华润三九：2022年年度报告.md


处理中华润三九：2022年年度报告.md: 100%|██████████| 296/296 [33:00<00:00,  6.69s/it] 


文件已存在: e:\github\TAIDIBEI_B\processed\reports-深交所\华润三九：2022年年度报告摘要.md


处理中华润三九：2022年年度报告摘要.md:   0%|          | 0/7 [00:03<?, ?it/s]


KeyboardInterrupt: 

[(WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2022年年度报告摘要.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2022年年度报告摘要.md')),
 (WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2023年一季度报告.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2023年一季度报告.md')),
 (WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2023年三季度报告.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2023年三季度报告.md')),
 (WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2023年半年度报告.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2023年半年度报告.md')),
 (WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2023年半年度报告摘要.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2023年半年度报告摘要.md')),
 (WindowsPath('e:/github/TAIDIBEI_B/parsed_results/reports-深交所/华润三九：2023年年度报告.md'),
  WindowsPath('e:/github/TAIDIBEI_B/processed/reports-深交所/华润三九：2023年年度报告.md')),


In [ ]:
from  multiprocessing import Process,Pool
import os, time, random

def fun1(name , z ):
    print('Run task %s (%s)...' % (name, os.getpid()))
    print(z)
    start = time.time()
    time.sleep(random.random())
    end = time.time()
    print('Task %s runs %0.2f seconds.' % (name, (end - start)))

if __name__=='__main__':
    pool = Pool(5) #创建一个5个进程的进程池

    for i in range(50):
        pool.apply_async(func=fun1, args=(i,'gadsgs'))

    pool.close()
    pool.join()
    print('结束测试')

In [ ]:
outer = tqdm(range(5), desc="外层", position=0)
for i in outer:
    inner = tqdm(range(100), desc=f"内层 {i+1}", position=1, leave=False)
    for j in inner:
        time.sleep(0.01)
        inner.set_postfix(current=j)
    inner.close()
outer.close()

外层: 100%|██████████| 5/5 [00:08<00:00,  1.73s/it]


In [26]:
import pandas as pd 
data = pd.read_excel('data/raw/示例数据/附件3：数据库-表名及字段说明.xlsx',sheet_name=None)
data.keys() 

dict_keys(['数据库表名', '核心业绩指标表', '资产负债表', '现金流量表', '利润表'])

In [29]:
DataBaseTableName = data['数据库表名']
CorePerformanceIndicator = data['核心业绩指标表']
Balance = data['资产负债表']
Income = data['利润表']
CashFlow= data['现金流量表']
    

In [31]:
print( CorePerformanceIndicator )

                                 字段名称               中文名称          字段类型   \
0                       serial_number                 序号            int   
1                          stock_code               股票代码    varchar(20)   
2                          stock_abbr               股票简称    varchar(50)   
3                                 eps            每股收益(元)  decimal(10,4)   
4             total_operating_revenue          营业总收入(万元)  decimal(20,2)   
5        operating_revenue_yoy_growth      营业总收入-同比增长(%)  decimal(10,4)   
6        operating_revenue_qoq_growth    营业总收入-季度环比增长(%)  decimal(10,4)   
7                 net_profit_10k_yuan            净利润(万元)  decimal(20,2)   
8               net_profit_yoy_growth        净利润-同比增长(%)  decimal(10,4)   
9               net_profit_qoq_growth      净利润-季度环比增长(%)  decimal(10,4)   
10                net_asset_per_share           每股净资产(元)  decimal(10,4)   
11                                roe          净资产收益率(%)  decimal(10,4)   
12             operating_

创建心业绩报表、资产负债表、现金流量表、利润表中

清洗数据